In [ ]:
import os
import re
import functools
import pandas as pd
import geopandas as gpd
import osmnx as ox
import folium
import pyogrio
from shapely.geometry import Point, MultiPoint

In [ ]:
OUTPUT_DIR = "parking_analysis_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Retrieve OSMnx data
# CRS unit will be degrees, so CRS needs to be converted into distance (feet) later

OUTPUT_DIR_network = "parking_analysis_outputs_network"
os.makedirs(OUTPUT_DIR_network, exist_ok=True)

bay_area_counties = [
    "Alameda County, California, USA", "Contra Costa County, California, USA", 
    "Marin County, California, USA", "Napa County, California, USA", 
    "San Francisco, California, USA", "San Mateo County, California, USA", 
    "Santa Clara County, California, USA", "Solano County, California, USA", 
    "Sonoma County, California, USA"
]

In [ ]:
def fetch_and_save_raw_network(county_name):
    safe_name = county_name.replace(", California, USA", "").replace(" ", "_").lower()
    raw_filepath = os.path.join(OUTPUT_DIR_network, f"{safe_name}_raw_network.gpkg")

    if os.path.exists(raw_filepath):
        print(f"Complete: {county_name} (found existing file)")
        return
        
    # tags = {'highway': ['residential', 'secondary', 'tertiary']} # Chester-Helmrich-Li Model excludes 'primary' and 'unclassified'
    tags = {'highway': ['residential', 'secondary', 'tertiary', 'primary', 'unclassified']} # 'primary' and 'unclassified' will be used for later model modification
    
    try:
        gdf = ox.features_from_place(county_name, tags)
        if gdf.empty:
            print(f"Complete: {county_name} (no data found)")
            return

        # Filter for valid geometries
        gdf = gdf[gdf.geometry.geom_type.isin(['LineString', 'MultiLineString'])]
        
        # Filter for and clean up columns
        essential_cols = ['highway', 'maxspeed', 'bridge', 'tunnel', 'geometry', 'name']
        cols_to_keep = [c for c in gdf.columns if c in essential_cols]
        gdf = gdf[cols_to_keep]

        gdf.columns = [c.replace(':', '_').lower() for c in gdf.columns]
        for col in gdf.columns:
            if col != 'geometry':
                gdf[col] = gdf[col].astype(str).replace('nan', None)

        # Save the retrieved network data
        gdf.to_file(raw_filepath, layer='raw_osm', driver="GPKG")
        print(f"Complete: {county_name}")
    
    except Exception as e:
        print(f"Error fetching {county_name}: {e}")

In [ ]:
# Fetch all counties' street networks
for county in bay_area_counties:
    fetch_and_save_raw_network(county)

In [ ]:
# Dataset overview
sample_file = os.path.join(OUTPUT_DIR_network, "san_francisco_raw_network.gpkg")

if os.path.exists(sample_file):
    gdf_sample = gpd.read_file(sample_file, layer='raw_osm')

    print("Sample GDF CRS:", gdf_sample.crs)
    
    print("Sample GDF Overview")
    display(gdf_sample.head())
    
    print("\nMaxspeed Column Breakdown")
    display(gdf_sample['maxspeed'].value_counts(dropna=False))
else:
    print("Run the fetch_and_save_raw_network function first.")

In [ ]:
def extract_incorporated_network(county_name):
    """
    Reads the raw network and saves two baseline files:
      _parking_FULL.gpkg        — all road types, whole county
      _parking_INCORPORATED.gpkg — residential/secondary/tertiary, incorporated cities only
    """
    safe_name = county_name.replace(", California, USA", "").replace(" ", "_").lower()
    raw_path = os.path.join(OUTPUT_DIR_network, f"{safe_name}_raw_network.gpkg")
    full_path = os.path.join(OUTPUT_DIR, f"{safe_name}_parking_FULL.gpkg")
    inc_path = os.path.join(OUTPUT_DIR, f"{safe_name}_parking_INCORPORATED.gpkg")
    
    if not os.path.exists(raw_path):
        print(f"Skipping {county_name}: Raw network not found.")
        return

    # Load the raw network (Comes in as EPSG:4326)
    full_roads = gpd.read_file(raw_path, layer='raw_osm')
    
    if "San Francisco" in county_name:
        # San Francisco is a consolidated city-county (admin_level 6). 
        # All roads in the county are "incorporated" city roads.
        inc_roads = full_roads.copy()
    else:
        # For other counties, get city boundaries for 'Incorporated' areas
        try:
            # admin_level 8 represents incorporated cities/towns in OSM
            cities = ox.features_from_place(county_name, tags={'admin_level': '8'})
            cities = cities[cities.geometry.type.isin(['Polygon', 'MultiPolygon'])]
            
            # Spatial join happens here as both layers are still in 4326
            inc_roads = gpd.sjoin(full_roads, cities[['geometry']], how='inner', predicate='intersects')
            if 'osmid' in inc_roads.columns:
                inc_roads = inc_roads.drop_duplicates(subset='osmid')
            else:
                inc_roads = inc_roads.drop_duplicates(subset='geometry')
            
        except Exception as e:
            print(f"Could not fetch city boundaries for {county_name}. Defaulting to full county.")
            inc_roads = full_roads.copy()

    allowed_inc_highways = ['residential', 'secondary', 'tertiary']
    inc_roads = inc_roads[inc_roads['highway'].isin(allowed_inc_highways)]
    
    # Project to CRS using feet (EPSG:2227)
    full_roads = full_roads.to_crs(epsg=2227)
    inc_roads = inc_roads.to_crs(epsg=2227)

    # Save the files 
    full_roads.to_file(full_path, layer='on_street_total', driver="GPKG")
    inc_roads.to_file(inc_path, layer='on_street_incorporated', driver="GPKG")
    print(f"Complete: {county_name} (Created FULL and INCORPORATED baseline geometries in EPSG:2227)")

In [ ]:
# Curb reduction constants — Chester-Helmrich-Li Model
CURB_REDUCTIONS = {
    'intersection_ft':   33.0,   # Per intersection
    'bus_stop_ft':       33.0,   # Per bus stop
    'driveway_ft':       15.0,   # Per driveway
    'hydrant_spacing_ft': 500.0, # 1 space lost per 500 ft of curb
    'daylighting_ft':    20.0    # Daylighting setback
}

In [ ]:
def add_osm_feature_counts(streets_gdf, county_name):
    """
    Fetches bus stops and driveways from OSM and counts how many
    intersect each street segment.
    """
    # Initialize counts at 0
    streets_gdf['bus_stop_count'] = 0
    streets_gdf['driveway_count'] = 0
    
    try:
        tags = {
            'highway': ['bus_stop'],
            'public_transport': ['platform', 'stop_position'],
            'service': ['driveway']
        }
        # Fetch and project to feet
        osm_features = ox.features_from_place(county_name, tags).to_crs(epsg=2227)
        if osm_features.empty: return streets_gdf
        
        # Separate the features and buffer them slightly (10ft) so they touch the road lines
        bus_stops = osm_features[(osm_features['highway'] == 'bus_stop') | (osm_features['public_transport'].notna())].copy()
        driveways = osm_features[osm_features['service'] == 'driveway'].copy()
        
        # Spatial Join to count intersections
        
        bus_stops['geometry'] = bus_stops.geometry.buffer(10)
        joined_bus = gpd.sjoin(bus_stops, streets_gdf, how='inner', predicate='intersects')
        bus_counts = joined_bus.groupby('index_right').size()
        streets_gdf.loc[bus_counts.index, 'bus_stop_count'] = bus_counts
    
    
        driveways['geometry'] = driveways.geometry.buffer(10)
        joined_dw = gpd.sjoin(driveways, streets_gdf, how='inner', predicate='intersects')
        dw_counts = joined_dw.groupby('index_right').size()
        streets_gdf.loc[dw_counts.index, 'driveway_count'] = dw_counts
            
    except Exception as e:
        print(f"Could not fetch OSM features for {county_name} - {e}")
        
    return streets_gdf

In [ ]:
# Replication

def calculate_spots(row, county_name, daylighting=False, include_high_speed=False):
    # Set parking spot sizes based on location
    # In Chester-Helmrich-Li Model, San Francisco uses a shorter spot length estimation due to 50 road-miles of perpendicular on-street parking
    is_sf = "San Francisco" in county_name
    spot_length_ft = 18.4 if is_sf else 20.0
    
    # 1. Filter out non-parkable infrastructure
    if any([str(row.get('bridge', 'no')).lower() == 'yes',
            str(row.get('tunnel', 'no')).lower() == 'yes',
            str(row.get('highway', '')).lower() == 'motorway']):
        return 0

    # 2. Filter out high-speed roads (>= 45 mph) that are not in Chester-Helmrich-Li Model
    maxspeed_val = str(row.get('maxspeed', '0'))
    speeds = re.findall(r'\d+', maxspeed_val)
    if not include_high_speed and speeds and int(speeds[0]) >= 45:
        return 0

    # 3. Calculate usable curb space (Replacing the parkable_multiplier)
    usable_length_ft = row.geometry.length
    
    # Intersection deduction
    usable_length_ft -= CURB_REDUCTIONS['intersection_ft']
    
    # Deduct actual OSM bus stops and driveways found on this specific segment
    usable_length_ft -= (row.get('bus_stop_count', 0) * CURB_REDUCTIONS['bus_stop_ft'])
    usable_length_ft -= (row.get('driveway_count', 0) * CURB_REDUCTIONS['driveway_ft'])

    # 4. Fire Hydrant Deduction (subtract 1 parking space per 500 ft of curb space)
    if usable_length_ft > 0:
        hydrant_deduction_ft = (usable_length_ft / CURB_REDUCTIONS['hydrant_spacing_ft']) * spot_length_ft
        usable_length_ft -= hydrant_deduction_ft

    # 5. Deduct Daylighting spots
    if daylighting:
        usable_length_ft -= CURB_REDUCTIONS['daylighting_ft']

    # Return Results
    if usable_length_ft <= 0: 
        return 0
    return int((usable_length_ft / spot_length_ft) * 2)

In [ ]:
def run_parking_calc(county_name, mode='INCORPORATED'):
    """
    mode='INCORPORATED' — Chester-Helmrich-Li replication:
                          incorporated cities, residential/secondary/tertiary, excludes >= 45 mph
    mode='FULL'         — Extended model:
                          whole county, all road types, includes >= 45 mph
    """
    include_high_speed = (mode == 'FULL')
    layer_name = 'on_street_incorporated' if mode == 'INCORPORATED' else 'on_street_total'

    safe_name = county_name.replace(", California, USA", "").replace(" ", "_").lower()
    input_path  = os.path.join(OUTPUT_DIR, f"{safe_name}_parking_{mode}.gpkg")
    output_path = os.path.join(OUTPUT_DIR, f"{safe_name}_parking_{mode}_model.gpkg")

    if not os.path.exists(input_path):
        return {"base": 0, "daylight": 0}

    gdf = gpd.read_file(input_path).to_crs(epsg=2227)

    # Apply OSM Bus Stop and Driveway Deductions
    print(f"Counting bus stops and driveways for {county_name}...")
    gdf = add_osm_feature_counts(gdf, county_name)

    # Calculate both columns
    gdf['replicated_base']     = gdf.apply(lambda r: calculate_spots(r, county_name, daylighting=False, include_high_speed=include_high_speed), axis=1)
    gdf['replicated_daylight'] = gdf.apply(lambda r: calculate_spots(r, county_name, daylighting=True,  include_high_speed=include_high_speed), axis=1)

    # Save to new files
    gdf.to_file(output_path, layer=layer_name, driver="GPKG")

    return {
        "base":     int(gdf['replicated_base'].sum()),
        "daylight": int(gdf['replicated_daylight'].sum())
    }

In [ ]:
calc_incorporated = functools.partial(run_parking_calc, mode='INCORPORATED')

# Modified Replication: Include Unincorporated land and roads at and above 45 mph
calc_full = functools.partial(run_parking_calc, mode='FULL')

In [ ]:
sf_name = "San Francisco, California, USA"
extract_incorporated_network(sf_name)
run_parking_calc(sf_name, mode='INCORPORATED')
run_parking_calc(sf_name, mode='FULL')

In [ ]:
# Summary Table preparation

# Chester-Helmrich-Li model Bay Area on-street parking modeling results
chester_results = {
    "Alameda":       1628432,
    "Contra Costa":  1462758,
    "Marin":          508014,
    "Napa":           395305,
    "San Francisco":  313988,
    "San Mateo":      768748,
    "Santa Clara":   1600879,
    "Solano":         933662,
    "Sonoma":        1024859
}

In [ ]:
def generate_parking_summary(county_list, chester_data, calc_func):
    """
    Runs calc_func for every county and returns a comparison table
    against Chester-Helmrich-Li published results.
    """
    results_data = []
    for county in county_list:
        clean_name = county.split(',')[0].replace(" County", "")
        
        stats = calc_func(county)  # calc_func handles everything now
        
        base = stats['base']
        dayl = stats['daylight']
        ches = chester_data.get(clean_name, 0)
        
        pct_daylight_impact = ((dayl - base) / base * 100) if base > 0 else 0
        pct_base_vs_chester = ((base - ches) / ches * 100) if ches > 0 else 0
        pct_dayl_vs_chester = ((dayl - ches) / ches * 100) if ches > 0 else 0
        
        results_data.append({
            "County": clean_name,
            "Chester-Helmrich-Li Model Published Results": ches,
            "Replicated Model": base,
            "Replication v.s. Chester-Helmrich-Li Model": f"{pct_base_vs_chester:.2f}%",
            "Replicated Model with Daylighting estimation": dayl,
            "Daylighting: Estimated % Reduction": f"{pct_daylight_impact:.2f}%",
            "Daylighting estimation v.s. Chester-Helmrich-Li Model": f"{pct_dayl_vs_chester:.2f}%"
        })

    df_final = pd.DataFrame(results_data)

    tot_base = df_final["Replicated Model"].sum()
    tot_dayl = df_final["Replicated Model with Daylighting estimation"].sum()
    tot_ches = df_final["Chester-Helmrich-Li Model Published Results"].sum()

    tot_pct_daylight    = ((tot_dayl - tot_base) / tot_base * 100) if tot_base > 0 else 0
    tot_pct_base_vs_ches = ((tot_base - tot_ches) / tot_ches * 100) if tot_ches > 0 else 0
    tot_pct_dayl_vs_ches = ((tot_dayl - tot_ches) / tot_ches * 100) if tot_ches > 0 else 0

    totals = {
        "County": "TOTAL",
        "Chester-Helmrich-Li Model Published Results": tot_ches,
        "Replicated Model": tot_base,
        "Replication v.s. Chester-Helmrich-Li Model": f"{tot_pct_base_vs_ches:.2f}%",
        "Replicated Model with Daylighting estimation": tot_dayl,
        "Daylighting: Estimated % Reduction": f"{tot_pct_daylight:.2f}%",
        "Daylighting estimation v.s. Chester-Helmrich-Li Model": f"{tot_pct_dayl_vs_ches:.2f}%"
    }

    df_final = pd.concat([df_final, pd.DataFrame([totals])], ignore_index=True)

    return df_final.style.format({
        "Chester-Helmrich-Li Model Published Results": "{:,}",
        "Replicated Model": "{:,}",
        "Replicated Model with Daylighting estimation": "{:,}"
    }).hide(axis='index')

In [ ]:
# Extract incorporated street networks for all counties
for county in bay_area_counties:
    extract_incorporated_network(county)

# Chester-Helmrich-Li Replication (incorporated, excludes primary road, no high-speed) + Daylighting
display(generate_parking_summary(bay_area_counties, chester_results,
                                  calc_func=calc_incorporated))

# Chester-Helmrich-Li Replication + Daylighting + Modification Attempt (full county, all road types, includes high-speed)
display(generate_parking_summary(bay_area_counties, chester_results,
                                  calc_func=calc_full))

In [ ]:
# Prepare for mapping on-street parking counts

def get_bbox_poly(lat1, lon1, lat2, lon2, buffer_feet=600):
    """
    Builds a bounding box polygon from two GPS coordinate pairs.
    Returns the polygon and its bounds, both in EPSG:2227 (feet).
    """
    # Create the point in standard GPS (4326)
    pts = gpd.GeoSeries([MultiPoint([Point(lon1, lat1), Point(lon2, lat2)])], crs="EPSG:4326")
    
    # Project to California Zone 3 Feet (2227) to make sure
    pts_2227 = pts.to_crs(epsg=2227)
    
    # Buffer in 2227
    bbox_poly = pts_2227.buffer(buffer_feet).envelope.iloc[0]
    
    return bbox_poly, bbox_poly.bounds

In [ ]:
def plot_neighborhood_dashboard(county_name, lat1, lon1, lat2, lon2, buffer_feet=600, use_incorporated=True):
    """
    Make an interactive folium map for any street segment level case study.
    Click on segments to accumulate spot counts in the dashboard panel.
    """
    base_name = county_name.lower().replace(" ", "_").replace("_county", "")
    suffix = "INCORPORATED" if use_incorporated else "FULL"
    
    potential_files = [f"{base_name}_parking_{suffix}_model.gpkg", f"{base_name}_county_parking_{suffix}_model.gpkg"]
    filepath = next((os.path.join(OUTPUT_DIR, f) for f in potential_files if os.path.exists(os.path.join(OUTPUT_DIR, f))), None)
    
    if not filepath: return "Error: No GPKG found."

    available_layers = pyogrio.list_layers(filepath)[:, 0] # Returns list of layer names
    
    # Try to find the intended layer, but fall back to the first available one if missing
    preferred_layer = 'on_street_incorporated' if use_incorporated else 'on_street_total'
    layer_name = preferred_layer if preferred_layer in available_layers else available_layers[0]
    
    bbox_poly, bbox_bounds = get_bbox_poly(lat1, lon1, lat2, lon2, buffer_feet)
    
    # Math Section (EPSG:2227)
    # Load and immediately project to Feet
    gdf = gpd.read_file(filepath, layer=layer_name, bbox=bbox_bounds).to_crs(epsg=2227)
    if gdf.empty: return "No data found."
    
    gdf['orig_segment_len'] = gdf.geometry.length
    noded_geom = gdf.geometry.union_all()
    
    lines = list(noded_geom.geoms) if hasattr(noded_geom, 'geoms') else [noded_geom]
    blocks_gdf = gpd.GeoDataFrame({'geometry': lines}, crs=2227)

    # Use 0.5 foot buffer for robust intersection
    blocks_gdf['midpoint'] = blocks_gdf.geometry.interpolate(0.5, normalized=True)
    gdf_temp = gdf.copy()
    gdf_temp.geometry = gdf.geometry.buffer(0.5) 
    
    joined = gpd.sjoin(blocks_gdf.set_geometry('midpoint'), gdf_temp, how='inner', predicate='within')
    final_gdf = joined.set_geometry(blocks_gdf.loc[joined.index, 'geometry']).drop(columns=['midpoint', 'index_right'])
    
    if final_gdf.empty: return "No valid street segments found after intersection."
    
    ratio = final_gdf.geometry.length / final_gdf['orig_segment_len']
    base_col = next((c for c in ['replicated_base', 'mti_spots', 'supply_est'] if c in final_gdf.columns), None)
    if base_col:
        final_gdf['Baseline Spots'] = (final_gdf[base_col] * ratio).round().astype(int)
    
    # Setup Tooltips
    tooltip_fields = ['name', 'Baseline Spots']
    tooltip_aliases = ['Street:', 'Baseline Model:']

    daylight_col = 'replicated_daylight' if 'replicated_daylight' in final_gdf.columns else 'mti_spots_daylighting'
    if daylight_col in final_gdf.columns:
        final_gdf['Daylight Spots'] = (final_gdf[daylight_col] * ratio).round().astype(int)
        final_gdf['Spots Lost'] = final_gdf['Baseline Spots'] - final_gdf['Daylight Spots']
        tooltip_fields.extend(['Daylight Spots', 'Spots Lost'])
        tooltip_aliases.extend(['After Daylighting:', 'Spots Lost:'])

    # Mapping Section (EPSG:4326)
    final_gdf_gps = final_gdf.to_crs(epsg=4326)
    
    # Force the center point
    center_lat, center_lon = (lat1 + lat2) / 2, (lon1 + lon2) / 2
    
    # Initialize map with location AND zoom_start locked in
    m = folium.Map(
        location=[center_lat, center_lon], 
        zoom_start=17, 
        tiles="cartodbpositron",
        zoom_control=True
    )

    bbox_gps = gpd.GeoSeries([bbox_poly], crs=2227).to_crs(epsg=4326).iloc[0]
    b = bbox_gps.bounds # [minx, miny, maxx, maxy]
    sw = [b[1], b[0]] # SouthWest
    ne = [b[3], b[2]] # NorthEast
    
    folium.GeoJson(
        final_gdf_gps, 
        style_function=lambda x: {'color': '#34495e', 'weight': 6, 'opacity': 0.6},
        tooltip=folium.GeoJsonTooltip(fields=tooltip_fields, aliases=tooltip_aliases)
    ).add_to(m)

    m.fit_bounds([sw, ne], max_zoom=18)
    
    # Add Node Markers
    for pt in [Point(c) for line in lines for c in [line.coords[0], line.coords[-1]]]:
        p_gps = gpd.GeoSeries([pt], crs=2227).to_crs(epsg=4326).iloc[0]
        folium.CircleMarker(
            location=[p_gps.y, p_gps.x], 
            radius=3.5, color='black', fill=True, fill_color='black', fill_opacity=1.0
        ).add_to(m)
        
    folium.Marker([lat1, lon1], icon=folium.Icon(color="red", icon="star")).add_to(m)
    folium.Marker([lat2, lon2], icon=folium.Icon(color="blue", icon="star")).add_to(m)

    # Dashboard Section
    dashboard_html = """
    <div id="selection-dashboard" style="
        position: fixed; top: 10px; right: 10px; width: 230px; 
        background-color: white; border: 2px solid grey; z-index: 9999; 
        font-family: Arial; font-size: 14px; padding: 15px;
        box-shadow: 3px 3px 5px rgba(0,0,0,0.3);">
        <h4 style="margin-top: 0; margin-bottom: 10px; text-align: center;">Selected Segments</h4>
        <b>Segments:</b> <span id="dash-count">0</span><br>
        <b>Baseline:</b> <span id="dash-base">0</span><br>
        <b>Daylight:</b> <span id="dash-day">0</span><br>
        <b>Spots Lost:</b> <span style="color:red;" id="dash-lost">0</span><br><br>
        <button onclick="location.reload()" style="width: 100%; padding: 5px; cursor: pointer;">Reset Map</button>
    </div>
    <script>
    var selectedLayers = new Set();
    var dashTotals = { count: 0, base: 0, day: 0, lost: 0 };
    function updateUI() {
        document.getElementById('dash-count').innerText = dashTotals.count;
        document.getElementById('dash-base').innerText = dashTotals.base;
        document.getElementById('dash-day').innerText = dashTotals.day;
        document.getElementById('dash-lost').innerText = dashTotals.lost;
    }
    setTimeout(function() {
        var mapKeys = Object.keys(window).filter(key => key.startsWith('map_'));
        if(mapKeys.length === 0) return;
        var mapObj = window[mapKeys[0]];
        mapObj.eachLayer(function(layer) {
            if (layer.feature && layer.feature.properties && layer.feature.properties['Baseline Spots'] !== undefined) {
                layer.on('click', function(e) {
                    var p = layer.feature.properties;
                    var lid = layer._leaflet_id;
                    if (selectedLayers.has(lid)) {
                        selectedLayers.delete(lid);
                        layer.setStyle({color: '#34495e', weight: 6, opacity: 0.6});
                        dashTotals.count--; dashTotals.base -= p['Baseline Spots'];
                        dashTotals.day -= (p['Daylight Spots'] || 0); dashTotals.lost -= (p['Spots Lost'] || 0);
                    } else {
                        selectedLayers.add(lid);
                        layer.setStyle({color: '#e74c3c', weight: 8, opacity: 1.0});
                        dashTotals.count++; dashTotals.base += p['Baseline Spots'];
                        dashTotals.day += (p['Daylight Spots'] || 0); dashTotals.lost += (p['Spots Lost'] || 0);
                    }
                    updateUI();
                });
            }
        });
    }, 1000);
    </script>
    """
    m.get_root().html.add_child(folium.Element(dashboard_html))

    return m

In [ ]:
# Case Study: San Francisco
# Mission Street (16th Street to 24th Street), San Francisco County

plot_neighborhood_dashboard(
    county_name="San Francisco", 
    lat1=37.765067, lon1=-122.419709,
    lat2=37.752255, lon2=-122.418415,
)

In [ ]:
# Case Study: San Francisco (Using FULL model)
plot_neighborhood_dashboard(
    county_name="San Francisco", 
    lat1=37.765067, lon1=-122.419709,
    lat2=37.752255, lon2=-122.418415,
    use_incorporated=False
)

In [ ]:
# Case Study: San Jose
# Alum Rock Avenue (North Capitol Avenue to South King Road), San Jose, Santa Clara County

plot_neighborhood_dashboard(
    county_name="Santa Clara", 
    lat1=37.362282, lon1=-121.835424,
    lat2=37.352908, lon2=-121.855596,
)

In [ ]:
# Case Study: San Jose (Using FULL model)

plot_neighborhood_dashboard(
    county_name="Santa Clara", 
    lat1=37.362282, lon1=-121.835424,
    lat2=37.352908, lon2=-121.855596,
    use_incorporated=False
)

In [ ]:
# Case Study: Oakland
# Grand Avenue (Euclid Avenue to Harrison Street), Oakland, Alameda County
plot_neighborhood_dashboard(
    county_name="Alameda", 
    lat1=37.808618, lon1=-122.251673,
    lat2=37.810931, lon2=-122.262382,
)

In [ ]:
# Case Study: Oakland (Using FULL model)
# Grand Avenue (Euclid Avenue to Harrison Street), Oakland, Alameda County
plot_neighborhood_dashboard(
    county_name="Alameda", 
    lat1=37.808618, lon1=-122.251673,
    lat2=37.810931, lon2=-122.262382,
    use_incorporated=False
)

In [ ]:
# Case Study: Walnut Creek
# Downtown Main Street (Civic Drive to Olympic Boulevard), Walnut Creek, Contra Costa County

plot_neighborhood_dashboard(
    county_name="Contra Costa", 
    lat1=37.901641, lon1=-122.061885,
    lat2=37.896310, lon2=-122.059948,
)

In [ ]:
# Case Study: Walnut Creek (Using FULL model)
plot_neighborhood_dashboard(
    county_name="Contra Costa", 
    lat1=37.901641, lon1=-122.061885,
    lat2=37.896310, lon2=-122.059948,
    use_incorporated=False
)